# 🚀 Production-Grade Machine Learning Pipeline for Credit Score Classification
## CodeAlpha Internship — Task 1: Credit Scoring Model

### 1. Library Imports & Global Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option('display.max_columns', None)

### 2. Loading Datasets

In [ ]:
train_df = pd.read_csv('train.csv', low_memory=False)
test_df = pd.read_csv('test.csv', low_memory=False)

In [ ]:
print(f"Train Dimensions: {train_df.shape}")
print(f"Test Dimensions:  {test_df.shape}")

### 3. Data Sanitization & Helper Functions

In [ ]:
def sanitize_numeric(val):
    if pd.isna(val):
        return np.nan
    clean_str = str(val).strip().replace('_', '')
    try:
        parsed = float(clean_str)
        return parsed if parsed >= 0 else np.nan
    except:
        return np.nan

In [ ]:
def parse_credit_history_months(val):
    if pd.isna(val) or val == '_':
        return np.nan
    match = re.search(r'(\d+)\s*Years\s*and\s*(\d+)\s*Months', str(val))
    if match:
        return int(match.group(1)) * 12 + int(match.group(2))
    return np.nan

In [ ]:
numeric_corrupted_cols = [
    'Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment',
    'Changed_Credit_Limit', 'Outstanding_Debt', 'Amount_invested_monthly', 'Monthly_Balance'
]

for df in [train_df, test_df]:
    for col in numeric_corrupted_cols:
        df[col] = df[col].apply(sanitize_numeric)
    df['Credit_History_Age_Months'] = df['Credit_History_Age'].apply(parse_credit_history_months)
    df['Occupation'] = df['Occupation'].replace('_______', np.nan)
    df['Credit_Mix'] = df['Credit_Mix'].replace('_', np.nan)
    df['Payment_of_Min_Amount'] = df['Payment_of_Min_Amount'].replace('NM', np.nan)

### 4. Advanced Financial Domain Feature Engineering

In [ ]:
for df in [train_df, test_df]:
    df['Debt_to_Income_Ratio'] = df['Outstanding_Debt'] / ((df['Annual_Income'] / 12) + 1e-5)
    df['EMI_to_Salary_Ratio'] = df['Total_EMI_per_month'] / (df['Monthly_Inhand_Salary'] + 1e-5)
    df['Total_Monthly_Outflow'] = df['Total_EMI_per_month'] + df['Amount_invested_monthly']
    df['Saving_Ratio'] = df['Monthly_Balance'] / (df['Monthly_Inhand_Salary'] + 1e-5)
    df['Debt_per_Loan'] = df['Outstanding_Debt'] / (df['Num_of_Loan'] + 1)
    df['Delayed_Payment_Per_Bank'] = df['Num_of_Delayed_Payment'] / (df['Num_Bank_Accounts'] + 1)
    df['Credit_Utilization_Per_Card'] = df['Credit_Utilization_Ratio'] / (df['Num_Credit_Card'] + 1)
    df['Inquiries_per_Bank_Account'] = df['Num_Credit_Inquiries'] / (df['Num_Bank_Accounts'] + 1)

In [ ]:
for df in [train_df, test_df]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

### 5. Defining Feature Vectors

In [ ]:
feature_cols = [
    'Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
    'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date',
    'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries',
    'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Total_EMI_per_month',
    'Amount_invested_monthly', 'Monthly_Balance', 'Credit_History_Age_Months',
    'Debt_to_Income_Ratio', 'EMI_to_Salary_Ratio', 'Total_Monthly_Outflow',
    'Saving_Ratio', 'Debt_per_Loan', 'Delayed_Payment_Per_Bank',
    'Credit_Utilization_Per_Card', 'Inquiries_per_Bank_Account'
]

In [ ]:
cat_cols = ['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']

### 6. Imputation Pipeline

In [ ]:
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

In [ ]:
train_df[feature_cols] = num_imputer.fit_transform(train_df[feature_cols])
test_df[feature_cols] = num_imputer.transform(test_df[feature_cols])

In [ ]:
train_df[cat_cols] = cat_imputer.fit_transform(train_df[cat_cols])
test_df[cat_cols] = cat_imputer.transform(test_df[cat_cols])

### 7. Categorical Encoding Pipeline

In [ ]:
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))

In [ ]:
all_features = feature_cols + cat_cols
target_le = LabelEncoder()
y = target_le.fit_transform(train_df['Credit_Score'])
X = train_df[all_features]
X_test_final = test_df[all_features]

In [ ]:
print(f"Final Feature Matrix Shape: {X.shape}")
print(f"Target Vector Shape: {y.shape}")

### 8. Stratified Train-Validation Splitting

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 9. Model Architecture Initialization

In [ ]:
rf_model = RandomForestClassifier(n_estimators=150, max_depth=16, random_state=42, n_jobs=-1)

In [ ]:
et_model = ExtraTreesClassifier(n_estimators=150, max_depth=16, random_state=42, n_jobs=-1)

In [ ]:
xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.08, max_depth=7, eval_metric='mlogloss', random_state=42, n_jobs=-1)

In [ ]:
lgb_model = LGBMClassifier(n_estimators=200, learning_rate=0.08, max_depth=8, random_state=42, n_jobs=-1)

### 10. Model Training Execution

In [ ]:
rf_model.fit(X_train, y_train)

In [ ]:
et_model.fit(X_train, y_train)

In [ ]:
xgb_model.fit(X_train, y_train)

In [ ]:
lgb_model.fit(X_train, y_train)

### 11. Model Evaluation & Benchmarking

In [ ]:
models_dict = {
    'Random Forest': rf_model,
    'Extra Trees': et_model,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model
}

In [ ]:
benchmark_metrics = []

for name, model in models_dict.items():
    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)
    
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average='macro')
    roc = roc_auc_score(y_val, probs, multi_class='ovr')
    
    benchmark_metrics.append({
        'Algorithm': name,
        'Accuracy': round(acc, 4),
        'Macro F1-Score': round(f1, 4),
        'ROC-AUC (OvR)': round(roc, 4)
    })

In [ ]:
summary_dataframe = pd.DataFrame(benchmark_metrics).sort_values(by='Macro F1-Score', ascending=False)
summary_dataframe

### 12. Best Model Classification Report

In [ ]:
champion_model = lgb_model
champion_preds = champion_model.predict(X_val)

In [ ]:
print(classification_report(y_val, champion_preds, target_names=target_le.classes_))

### 13. Confusion Matrix Diagnostic Plot

In [ ]:
cm = confusion_matrix(y_val, champion_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_le.classes_, yticklabels=target_le.classes_,
            cbar=False)
plt.title('Confusion Matrix Diagnostics — LightGBM Classifier', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Class')
plt.ylabel('True Ground Truth Class')
plt.tight_layout()
plt.show()

### 14. Feature Importance Profiling

In [ ]:
importances = champion_model.feature_importances_
importance_df = pd.DataFrame({'Feature': all_features, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(15), palette='mako')
plt.title('Top 15 Feature Importances — Champion LightGBM Model', fontsize=14, fontweight='bold')
plt.xlabel('Feature Importance Score')
plt.ylabel('Engineered Feature')
plt.tight_layout()
plt.show()

### 15. Model Artifact Serialization & Predictions Export

In [ ]:
joblib.dump(champion_model, 'credit_scoring_best_lgbm.pkl')
joblib.dump(target_le, 'target_label_encoder.pkl')

In [ ]:
test_preds_encoded = champion_model.predict(X_test_final)
test_preds_labels = target_le.inverse_transform(test_preds_encoded)

In [ ]:
submission_df = pd.DataFrame({
    'ID': test_df['ID'],
    'Customer_ID': test_df['Customer_ID'],
    'Credit_Score_Predicted': test_preds_labels
})
submission_df.to_csv('final_credit_score_predictions.csv', index=False)

In [ ]:
print("Execution Completed Successfully!")